In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from datasets import load_dataset
import pandas as pd

# Load Emotion dataset
emotion_dataset = load_dataset("dair-ai/emotion")

# Convert splits to Pandas
train_df = pd.DataFrame(emotion_dataset["train"])
val_df = pd.DataFrame(emotion_dataset["validation"])
test_df = pd.DataFrame(emotion_dataset["test"])

print("Dataset loaded successfully!")

print("\nTrain shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nSample:")
print(train_df.head())

README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset loaded successfully!

Train shape: (16000, 2)
Validation shape: (2000, 2)
Test shape: (2000, 2)

Columns:
['text', 'label']

Sample:
                                                text  label
0                            i didnt feel humiliated      0
1  i can go from feeling so hopeless to so damned...      0
2   im grabbing a minute to post i feel greedy wrong      3
3  i am ever feeling nostalgic about the fireplac...      2
4                               i am feeling grouchy      3


In [3]:
# Check label distribution
print("Label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nLabel names:")
print(emotion_dataset["train"].features["label"])

# Show one example from each label
print("\nExamples from each label:")

for label in sorted(train_df["label"].unique()):
    example = train_df[train_df["label"] == label].iloc[0]
    print(f"\nLabel {label}:")
    print(example["text"])

Label distribution:
label
0    4666
1    5362
2    1304
3    2159
4    1937
5     572
Name: count, dtype: int64

Label names:
ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])

Examples from each label:

Label 0:
i didnt feel humiliated

Label 1:
i have been with petronas for years i feel that petronas has performed well and made a huge profit

Label 2:
i am ever feeling nostalgic about the fireplace i will know that it is still on the property

Label 3:
im grabbing a minute to post i feel greedy wrong

Label 4:
i feel as confused about life as a teenager or as jaded as a year old man

Label 5:
ive been taking or milligrams or times recommended amount and ive fallen asleep a lot faster but i also feel like so funny


In [4]:
# Emotion label mapping

label_names = emotion_dataset["train"].features["label"].names

print("Emotion mapping:")
for i, name in enumerate(label_names):
    print(f"{i} -> {name}")

# Make copies
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

# Create readable emotion names
train_df["emotion"] = train_df["label"].map(dict(enumerate(label_names)))
val_df["emotion"] = val_df["label"].map(dict(enumerate(label_names)))
test_df["emotion"] = test_df["label"].map(dict(enumerate(label_names)))

print("\nSample after mapping:")
print(train_df.head())

Emotion mapping:
0 -> sadness
1 -> joy
2 -> love
3 -> anger
4 -> fear
5 -> surprise

Sample after mapping:
                                                text  label  emotion
0                            i didnt feel humiliated      0  sadness
1  i can go from feeling so hopeless to so damned...      0  sadness
2   im grabbing a minute to post i feel greedy wrong      3    anger
3  i am ever feeling nostalgic about the fireplac...      2     love
4                               i am feeling grouchy      3    anger


In [5]:
from datasets import Dataset

# Keep only the columns needed for training
train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print("Train dataset:", train_dataset)
print("Validation dataset:", val_dataset)
print("Test dataset:", test_dataset)

Train dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 16000
})
Validation dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})
Test dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})


In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 6

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load pretrained Transformer with a 6-class classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

print("Model and tokenizer loaded successfully!")
print("Model:", MODEL_NAME)
print("Number of labels:", NUM_LABELS)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model and tokenizer loaded successfully!
Model: distilbert-base-uncased
Number of labels: 6


In [7]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

# Tokenize datasets
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed!")

print("\nTrain features:")
print(tokenized_train.column_names)

print("\nExample:")
print(tokenized_train[0])

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenization completed!

Train features:
['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']

Example:
{'text': 'i didnt feel humiliated', 'label': 0, 'input_ids': [101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_

In [8]:
# Prepare the already-tokenized datasets for Trainer

train_tokenized = tokenized_train.rename_column("label", "labels")
val_tokenized = tokenized_val.rename_column("label", "labels")
test_tokenized = tokenized_test.rename_column("label", "labels")

print("Train features:")
print(train_tokenized.column_names)

print("\nValidation features:")
print(val_tokenized.column_names)

print("\nTest features:")
print(test_tokenized.column_names)

Train features:
['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']

Validation features:
['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']

Test features:
['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [9]:
import transformers
import torch

print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)

Transformers version: 5.0.0
PyTorch version: 2.10.0+cu128


In [10]:
from transformers import Trainer

In [11]:
from transformers import Trainer, TrainingArguments

# 1. تعريف إعدادات التدريب
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized
)

print("Trainer is ready!")
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Trainer is ready!
GPU available: True
GPU: Tesla T4


In [13]:
print(train_tokenized.features)
print(train_tokenized[0])

{'text': Value('string'), 'labels': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
{'text': 'i didnt feel humiliated', 'labels': 0, 'input_ids': [101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [14]:
print("Starting Emotion Classifier training...")

train_result = trainer.train()

print("\nTraining completed successfully!")

Starting Emotion Classifier training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.496159,0.418912
2,0.303151,0.319530
3,0.199024,0.307340


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed successfully!


In [15]:
from sklearn.metrics import accuracy_score, classification_report

# Evaluate on the test set
test_results = trainer.predict(test_tokenized)

# Get predictions
test_predictions = test_results.predictions.argmax(axis=-1)

# True labels
test_labels = test_tokenized["labels"]

# Accuracy
test_accuracy = accuracy_score(test_labels, test_predictions)

print("Emotion Classifier - Final Test Accuracy:", round(test_accuracy, 4))

print("\nFinal Test Classification Report:")
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=[
            "sadness",
            "joy",
            "love",
            "anger",
            "fear",
            "surprise"
        ],
        digits=4
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Emotion Classifier - Final Test Accuracy: 0.9285

Final Test Classification Report:
              precision    recall  f1-score   support

     sadness     0.9559    0.9690    0.9624       581
         joy     0.9472    0.9554    0.9513       695
        love     0.8636    0.8365    0.8498       159
       anger     0.9401    0.9127    0.9262       275
        fear     0.8589    0.9241    0.8903       224
    surprise     0.8125    0.5909    0.6842        66

    accuracy                         0.9285      2000
   macro avg     0.8964    0.8648    0.8774      2000
weighted avg     0.9278    0.9285    0.9274      2000



In [16]:
# Save the trained Emotion model and tokenizer

import os

os.makedirs("models/emotion_classifier", exist_ok=True)

trainer.save_model("models/emotion_classifier")
tokenizer.save_pretrained("models/emotion_classifier")

print("Emotion model and tokenizer saved successfully!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Emotion model and tokenizer saved successfully!


In [17]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles/folders in working directory:")
print(os.listdir("."))

print("\nSearching for saved Emotion model files...")

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if (
            "config.json" in file
            or "model.safetensors" in file
            or "pytorch_model.bin" in file
            or "tokenizer" in file
        ):
            print(os.path.join(root, file))


Current directory:
/kaggle/working

Files/folders in working directory:
['results', '.virtual_documents', 'models']

Searching for saved Emotion model files...
/kaggle/working/results/checkpoint-3000/model.safetensors
/kaggle/working/results/checkpoint-3000/config.json
/kaggle/working/results/checkpoint-2000/model.safetensors
/kaggle/working/results/checkpoint-2000/config.json
/kaggle/working/results/checkpoint-1000/model.safetensors
/kaggle/working/results/checkpoint-1000/config.json
/kaggle/working/models/emotion_classifier/tokenizer.json
/kaggle/working/models/emotion_classifier/model.safetensors
/kaggle/working/models/emotion_classifier/config.json
/kaggle/working/models/emotion_classifier/tokenizer_config.json


In [18]:
# Save Emotion model as a Kaggle output

import os

save_path = "/kaggle/working/emotion_classifier"

os.makedirs(save_path, exist_ok=True)

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Saved Emotion files:")
for root, dirs, files in os.walk(save_path):
    for file in files:
        print(os.path.join(root, file))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Emotion files:
/kaggle/working/emotion_classifier/tokenizer.json
/kaggle/working/emotion_classifier/model.safetensors
/kaggle/working/emotion_classifier/config.json
/kaggle/working/emotion_classifier/training_args.bin
/kaggle/working/emotion_classifier/tokenizer_config.json
